# 🐠 Unified Marine Detector - GPU Training

Train a YOLOv5 model to detect:
- **Marine Life** (7 classes): fish, jellyfish, penguin, puffin, shark, starfish, stingray
- **Fish Species** (12 classes): surgeonfish, triggerfish, jack, spadefish, wrasse, snapper, angelfish, damselfish, parrotfish, tuna, grouper, moorish_idol
- **Fish Disease** (4 classes): bacterial_gill_disease, bacterial_red_disease, bacterial_disease, healthy_fish

**Total: 23 classes**

---

## Prerequisites
1. Upload `unified_dataset.zip` to your Google Drive
2. Enable GPU: Runtime → Change runtime type → T4 GPU

## Step 1: Check GPU Availability

In [ ]:
# Check if GPU is available
!nvidia-smi

import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Step 2: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 3: Clone YOLOv5 and Install Requirements

In [ ]:
# Clone YOLOv5 repository
!git clone https://github.com/ultralytics/yolov5.git
%cd yolov5

# Install requirements
!pip install -r requirements.txt -q

## Step 4: Extract Dataset

Make sure you uploaded `unified_dataset.zip` to your Google Drive root folder.

If it's in a different location, update the path below.

In [ ]:
import os

# Path to your zip file in Google Drive (update if needed)
zip_path = '/content/drive/MyDrive/unified_dataset.zip'

# Check if file exists
if os.path.exists(zip_path):
    print(f"✅ Found dataset at: {zip_path}")
else:
    print(f"❌ Dataset not found at: {zip_path}")
    print("\nPlease upload unified_dataset.zip to your Google Drive root folder.")
    print("Or update the zip_path variable with the correct path.")

In [ ]:
# Extract dataset to /content/ (use -o to overwrite without prompts)
!unzip -o -q "{zip_path}" -d /content/

# The zip extracts to data/unified_dataset/, so list that path
!ls -la /content/data/unified_dataset/

## Step 5: Create Dataset Configuration

Create a new dataset.yaml with Colab-compatible paths.

In [ ]:
# Create dataset.yaml with correct Colab paths
# NOTE: The zip extracts to /content/data/unified_dataset/
dataset_yaml = """
path: /content/data/unified_dataset
train: images/train
val: images/val

nc: 23
names:
  0: fish
  1: jellyfish
  2: penguin
  3: puffin
  4: shark
  5: starfish
  6: stingray
  7: surgeonfish
  8: triggerfish
  9: jack
  10: spadefish
  11: wrasse
  12: snapper
  13: angelfish
  14: damselfish
  15: parrotfish
  16: tuna
  17: grouper
  18: moorish_idol
  19: bacterial_gill_disease
  20: bacterial_red_disease
  21: bacterial_disease
  22: healthy_fish
"""

with open('/content/data/unified_dataset/dataset.yaml', 'w') as f:
    f.write(dataset_yaml)

print("✅ Created dataset.yaml")
!cat /content/data/unified_dataset/dataset.yaml

## Step 6: Verify Dataset Structure

In [ ]:
import os

# Check dataset structure (note the /data/ in the path)
base_path = '/content/data/unified_dataset'
train_images = f'{base_path}/images/train'
val_images = f'{base_path}/images/val'
train_labels = f'{base_path}/labels/train'
val_labels = f'{base_path}/labels/val'

print("Dataset Structure:")
print(f"  Train images: {len(os.listdir(train_images)) if os.path.exists(train_images) else 'NOT FOUND'}")
print(f"  Val images: {len(os.listdir(val_images)) if os.path.exists(val_images) else 'NOT FOUND'}")
print(f"  Train labels: {len(os.listdir(train_labels)) if os.path.exists(train_labels) else 'NOT FOUND'}")
print(f"  Val labels: {len(os.listdir(val_labels)) if os.path.exists(val_labels) else 'NOT FOUND'}")

## Step 7: Start Training 🚀

Training parameters:
- **Epochs**: 50 (adjust as needed)
- **Batch size**: 32 (GPU can handle more than CPU)
- **Image size**: 640
- **Model**: YOLOv5s (small, good balance of speed/accuracy)
- **Device**: 0 (GPU)

Estimated time: ~30-60 minutes on T4 GPU

In [ ]:
# Train the model
!python train.py \
    --data /content/data/unified_dataset/dataset.yaml \
    --epochs 50 \
    --batch-size 32 \
    --img 640 \
    --weights yolov5s.pt \
    --project /content/runs/train \
    --name unified_marine_detector \
    --device 0 \
    --cache

## Step 8: View Training Results

In [ ]:
# Display training results
from IPython.display import Image, display
import os

results_dir = '/content/runs/train/unified_marine_detector'

# Show results.png if it exists
results_img = os.path.join(results_dir, 'results.png')
if os.path.exists(results_img):
    print("Training Results:")
    display(Image(filename=results_img, width=800))

# Show confusion matrix
confusion_img = os.path.join(results_dir, 'confusion_matrix.png')
if os.path.exists(confusion_img):
    print("\nConfusion Matrix:")
    display(Image(filename=confusion_img, width=600))

## Step 9: Test the Model

In [ ]:
# Run inference on validation images
!python detect.py \
    --weights /content/runs/train/unified_marine_detector/weights/best.pt \
    --source /content/data/unified_dataset/images/val \
    --img 640 \
    --conf 0.25 \
    --save-txt \
    --project /content/runs/detect \
    --name test_results

In [ ]:
# Display some detection results
import glob
from IPython.display import Image, display

detect_results = glob.glob('/content/runs/detect/test_results/*.jpg')[:6]
print(f"Showing {len(detect_results)} detection results:\n")

for img_path in detect_results:
    display(Image(filename=img_path, width=400))
    print()

## Step 10: Save Model to Google Drive

In [ ]:
import shutil
import os

# Source paths
best_weights = '/content/runs/train/unified_marine_detector/weights/best.pt'
last_weights = '/content/runs/train/unified_marine_detector/weights/last.pt'

# Destination in Google Drive
drive_dest = '/content/drive/MyDrive/marine_detector_models'
os.makedirs(drive_dest, exist_ok=True)

# Copy weights
if os.path.exists(best_weights):
    shutil.copy(best_weights, os.path.join(drive_dest, 'unified_marine_detector.pt'))
    print(f"✅ Saved best.pt to: {drive_dest}/unified_marine_detector.pt")

if os.path.exists(last_weights):
    shutil.copy(last_weights, os.path.join(drive_dest, 'unified_marine_detector_last.pt'))
    print(f"✅ Saved last.pt to: {drive_dest}/unified_marine_detector_last.pt")

# Also copy the entire training run for reference
results_dir = '/content/runs/train/unified_marine_detector'
if os.path.exists(results_dir):
    shutil.copytree(results_dir, os.path.join(drive_dest, 'training_results'), dirs_exist_ok=True)
    print(f"✅ Saved training results to: {drive_dest}/training_results/")

print("\n📁 Files in Google Drive:")
!ls -la "{drive_dest}"

## Step 11: Download Model Directly (Alternative)

In [ ]:
# Download the best weights directly to your computer
from google.colab import files

best_weights = '/content/runs/train/unified_marine_detector/weights/best.pt'
if os.path.exists(best_weights):
    files.download(best_weights)
    print("\n✅ Download started! Save as 'unified_marine_detector.pt'")
else:
    print("❌ Model not found. Make sure training completed successfully.")

---

## 🎉 Done!

Your trained model is now saved to Google Drive at:
- `MyDrive/marine_detector_models/unified_marine_detector.pt`

### To use the model locally:

1. Download `unified_marine_detector.pt` from Google Drive
2. Place it in your project's `models/` folder
3. Run the backend: `python backend/main.py`
4. Run the frontend: `cd frontend-ui && npm run dev`

### Model Info:
- **Classes**: 23 (marine life, fish species, fish diseases)
- **Architecture**: YOLOv5s
- **Input size**: 640x640